# Pattern Recognition and Machine Learning

## Project

### Comparative Study of kNN and Weighted kNN Classification on the Palmer Penguins Dataset

#### Completed by: **Group 6**

| Index | Full Name |
| ----- | ----- |
| 66m/25 | Ivona Petrović |
| 144m/25 | Vojin Đorđević |

**Imports and data loading.** We use `pandas` for loading and manipulating the tabular dataset, `os` for file system operations, and `numpy` for saving the final feature/label arrays. `load_penguins_data` fetches the Palmer Penguins dataset from a public CSV source and we save an untouched raw copy to `../data/raw/penguins_.csv` before any cleaning.

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def load_penguins_data():
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
    df = pd.read_csv(url)
    
    print("Raw shape:", df.shape)
    
    return df

df = load_penguins_data()
df.to_csv("../data/raw/penguins_.csv", index=False)

**Inspecting the raw data.** We preview the first rows to check column names and types, count missing values per column, and look at the rows/indices where values are missing, to decide how to handle them.

In [ ]:
df.head()

In [ ]:
missing_counts = df.isnull().sum().to_frame(name="missing_count")
missing_counts

In [ ]:
rows_with_missing = df[df.isnull().any(axis=1)]
rows_with_missing

In [ ]:
missing_indices = df[df.isnull().any(axis=1)].index
missing_indices 

**Cleaning the data.** Since missing entries are a small fraction of the dataset, we drop them with `dropna()` rather than imputing, producing `df_clean`. Shapes before/after are printed to quantify how much data was lost.

In [ ]:
df_clean = df.dropna()

print("Before:", df.shape)
print("After:", df_clean.shape)

**Feature/target preparation and saving.** We select the four numeric measurements (bill length, bill depth, flipper length, body mass) as features `X`, and species as the target. Species names are mapped to integers $\{0,1,2\}$ to produce `y_encoded`, while `df_eda` is kept as an independent copy for later exploratory analysis. Finally, the cleaned dataset and feature/label arrays are saved to disk for use in subsequent notebook stages.

In [ ]:
X = df_clean[[ "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
y = df_clean["species"]

In [ ]:
label_map = {
    "Adelie": 0,
    "Chinstrap": 1,
    "Gentoo": 2
}

y_encoded = y.map(label_map)

df_eda = df_clean.copy()  # separate dataset for EDA (kept independent from modeling data)

In [ ]:
os.makedirs("../data", exist_ok=True)

df_clean.to_csv("../data/penguins_clean.csv", index=False)
np.save("../data/X.npy", X.values)
np.save("../data/y.npy", y_encoded.values)

## 3. Exploratory Data Analysis

We explore the cleaned dataset (`df_eda`) before modeling: summary statistics, class balance, feature correlations, distributions by species, a from-scratch PCA projection, and outlier inspection via z-scores. Figures are saved to `../figures`.

In [ ]:
feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
species_list = sorted(df_eda["species"].unique())
colors = {"Adelie": "tab:blue", "Chinstrap": "tab:orange", "Gentoo": "tab:green"}

**Summary statistics by species.** We compute mean, standard deviation, min, and max for each of the four numeric features, grouped by species, to get a quantitative sense of how the species differ before any modeling.

In [ ]:
summary_stats = df_eda.groupby("species")[feature_cols].agg(["mean", "std", "min", "max"])
summary_stats = summary_stats.round(2)
summary_stats

**Class distribution.** We plot the number of samples per species after cleaning, to check whether the dataset is balanced or skewed toward one class — class imbalance can affect both the choice of evaluation metric and kNN's neighbor voting.

In [ ]:
class_counts = df_eda["species"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values, color=[colors[s] for s in class_counts.index])
plt.title("Class distribution after cleaning")
plt.xlabel("Species")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("../figures/class_distribution.png", dpi=150)
plt.show()

print(class_counts)
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (largest/smallest class): {imbalance_ratio:.2f}")

**Note on class imbalance.** The dataset shows moderate class imbalance — Adelie (146 samples) has roughly 2.15× more samples than Chinstrap (68 samples), with Gentoo (119 samples) in between. This is accounted for during model evaluation, where overall accuracy alone may be misleading and per-class metrics (e.g., precision/recall) are also reported.

**Correlation heatmap.** We compute the pairwise Pearson correlation among the four numeric features to identify redundant or strongly related measurements, which is relevant both for interpreting the data and for understanding what PCA will later capture.

In [ ]:
def correlation_matrix(X):
    n_features = X.shape[1]
    corr = np.zeros((n_features, n_features))
    
    means = X.mean(axis=0)
    stds = X.std(axis=0)
    
    for i in range(n_features):
        for j in range(n_features):
            cov_ij = np.mean((X[:, i] - means[i]) * (X[:, j] - means[j]))
            corr[i, j] = cov_ij / (stds[i] * stds[j])
    
    return corr

X_features = df_eda[feature_cols].values
corr_array = correlation_matrix(X_features)

corr_matrix = pd.DataFrame(corr_array, index=feature_cols, columns=feature_cols).round(2)
corr_matrix

In [ ]:
corr_no_diag = corr_matrix.where(~np.eye(len(corr_matrix), dtype=bool))
strongest = corr_no_diag.abs().unstack().idxmax()
strongest_val = corr_matrix.loc[strongest]
print(f"Strongest correlation: {strongest[0]} vs {strongest[1]} (r = {strongest_val:.2f})")

**Interpretation.** The strongest correlation is between flipper_length_mm and body_mass_g (r = 0.87), indicating that larger-bodied penguins tend to have proportionally longer flippers. This strong positive relationship suggests these two features carry overlapping information, which is consistent with the high variance explained by the first principal component in the PCA projection below.

**Feature distributions by species.** We use overlapping histograms for each of the four features, split by species, to see how much the distributions overlap or separate — features with clearly separated distributions are likely to be more informative for classification.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, feat in zip(axes.flatten(), feature_cols):
    for s in species_list:
        ax.hist(df_eda[df_eda["species"] == s][feat], alpha=0.5, label=s, color=colors[s], bins=15)
    ax.set_title(feat)
    ax.legend()

plt.tight_layout()
plt.savefig("../figures/distributions_by_species.png", dpi=150)
plt.show()

**Pairwise feature relationships.** We plot the two most relevant feature pairs (selected based on the correlation analysis above), colored by species, to visually inspect class separability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

pairs = [("flipper_length_mm", "body_mass_g"), ("bill_length_mm", "bill_depth_mm")]

for ax, (x_feat, y_feat) in zip(axes, pairs):
    for s in species_list:
        subset = df_eda[df_eda["species"] == s]
        ax.scatter(subset[x_feat], subset[y_feat], color=colors[s], label=s, alpha=0.6)
    ax.set_xlabel(x_feat)
    ax.set_ylabel(y_feat)
    ax.legend()

plt.tight_layout()
plt.savefig("../figures/key_scatter_pairs.png", dpi=150)
plt.show()

**PCA projection.** We standardize the four features (zero mean, unit variance) and project them onto the first two principal components, to visualize class separability in 2D before modeling. We also report the explained variance ratio of each component.

In [ ]:
X = df_eda[feature_cols].values
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

cov_matrix = (X_scaled.T @ X_scaled) / X_scaled.shape[0]
eigvals, eigvecs = np.linalg.eigh(cov_matrix)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

X_pca = X_scaled @ eigvecs[:, :2]
explained_var = eigvals / eigvals.sum()

plt.figure(figsize=(6, 5))
for s, c in colors.items():
    mask = (df_eda["species"] == s).values
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=s, color=c, alpha=0.7)

plt.xlabel(f"PC1 ({explained_var[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({explained_var[1]*100:.1f}%)")
plt.title("PCA projection")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/pca_projection.png", dpi=150)
plt.show()

print("Explained variance ratio:", explained_var[:2])

**Outlier inspection via z-scores.** We compute per-feature z-scores on the standardized data and flag any sample with |z| > 3 on at least one feature as a potential outlier. We list the flagged rows and record an explicit decision on whether to keep or remove them.

In [ ]:
z_scores = pd.DataFrame(X_scaled, columns=feature_cols, index=df_eda.index)
outliers = df_eda[(z_scores.abs() > 3).any(axis=1)]

print(f"Flagged outliers (|z| > 3): {len(outliers)}")
outliers

**Decision on flagged outliers.** No samples exceeded |z| > 3 on any of the four features, indicating no extreme outliers in the cleaned dataset. No additional rows were removed beyond the missing-value cleaning already performed.